# Chapter 4: Discrete Behavior Cloning

This Colab follows the Chapter 4 manuscript from the multimodal regression trap through a trained, discrete SO-101 action policy. A GPU runtime is strongly recommended for the real backbone cells.

In [ ]:
# Colab setup: install the three public chapter packages from GitHub.
import subprocess
import sys

if 'google.colab' in sys.modules:
    organization = 'Large-Robotics-Models-From-Scratch'
    requirements = [
        f'lrm-ch02[data] @ git+https://github.com/{organization}/lrm-code-chapter-2.git@main',
        f'lrm-ch03 @ git+https://github.com/{organization}/lrm-code-chapter-3.git@main',
        f'lrm-ch04[data] @ git+https://github.com/{organization}/lrm-code-chapter-4.git@main',
    ]
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *requirements],
        text=True, capture_output=True,
    )
    if result.returncode:
        detail = '\n'.join(
            part for part in (result.stdout, result.stderr) if part
        )
        raise RuntimeError(
            f'Chapter package installation failed:\n{detail}'
        )
    # LeRobot may upgrade Colab's preinstalled torch without upgrading
    # its optional torchaudio wheel. Transformers detects torchaudio by
    # package presence, then imports the incompatible binary while loading
    # SigLIP. Probe in a child process so a failed import cannot taint this
    # kernel; Chapters 2-4 do not use audio, so remove only a broken wheel.
    audio_probe = subprocess.run(
        [sys.executable, '-c', 'import torch, torchaudio'],
        text=True, capture_output=True,
    )
    if audio_probe.returncode:
        uninstall = subprocess.run(
            [sys.executable, '-m', 'pip', 'uninstall', '--yes',
             'torchaudio'],
            text=True, capture_output=True,
        )
        if uninstall.returncode:
            detail = '\n'.join(
                part for part in (uninstall.stdout, uninstall.stderr)
                if part
            )
            raise RuntimeError(
                f'Could not remove incompatible torchaudio:\n{detail}'
            )
        print('Removed an incompatible optional torchaudio wheel.')
    print('Installed Chapter 2, Chapter 3, and Chapter 4 packages.')

In [ ]:
import math
import numpy as np
import torch
import matplotlib.pyplot as plt

from ch04.constants import ACTION_BINS, ACTION_DIM, ACTION_HORIZON

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print(f'label grid: H={ACTION_HORIZON}, D={ACTION_DIM}, B={ACTION_BINS}')

## 4.2 The multimodal regression trap

The observation contains no clue about which of two equally valid expert modes was chosen. MSE therefore learns the conditional mean: zero, where the demonstrations have almost no density.

In [ ]:
from ch04.exercises import make_bimodal_actions, train_mse_baseline

observations, expert_actions = make_bimodal_actions()
mse_model, mse_history = train_mse_baseline()
with torch.no_grad():
    collapsed = mse_model(torch.zeros(1, 1)).item()
print(f'MSE prediction: {collapsed:+.3f} (expert modes are -1 and +1)')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(expert_actions.numpy(), bins=40)
axes[0].axvline(collapsed, color='crimson', label='MSE prediction')
axes[0].legend(); axes[0].set_xlabel('action')
axes[1].plot(mse_history); axes[1].set(xlabel='step', ylabel='MSE')
plt.show()

## 4.3 Fit the action tokenizer

We compute normalization statistics from training episodes only, then fit per-joint q01/q99 limits in that normalized space. The helper reads state and action columns directly from Arrow, so the full fit does not decode the camera videos.

In [ ]:
from ch04 import ActionTokenizer
from ch04.data import (DEFAULT_DATASET_ID, collect_normalized_actions,
                         make_chunked_dataloaders)

train_loader, validation_loader, stats = make_chunked_dataloaders(
    DEFAULT_DATASET_ID, batch_size=4, validation_fraction=0.1)
# None uses every training frame through the fast Arrow action column.
# Set an integer only for a bounded loader-based smoke fit.
TOKENIZER_FIT_BATCHES = None
normalized_actions = collect_normalized_actions(
    train_loader, stats, max_batches=TOKENIZER_FIT_BATCHES)
tokenizer = ActionTokenizer.fit(normalized_actions)

example = normalized_actions[0]
bins = tokenizer.encode(example)
decoded = tokenizer.decode(bins)
print('bins:   ', bins)
print('AR action embedding ids:', bins, '(separate 256-entry table)')
print('max normalized round-trip error:', np.abs(decoded-example).max())
print('tokenizer fit batches:', TOKENIZER_FIT_BATCHES or 'all')
print('train episodes:', train_loader.dataset.episodes)
print('validation episodes:', validation_loader.dataset.episodes)

In [ ]:
# A chunk has H vector-valued action positions and H x D labels.
demo_grid = torch.arange(ACTION_HORIZON * ACTION_DIM).reshape(
    1, ACTION_HORIZON, ACTION_DIM)
print('target grid:', tuple(demo_grid.shape))
print('action positions:', ACTION_HORIZON)
print('first two vectors:', demo_grid[0, :2].tolist())

## 4.4 Build the three action heads

The factorized head is the one-shot baseline, the autoregressive head is the exact chain-rule model, and the bidirectional parallel head is the manuscript's one-pass training and evaluation path.

In [ ]:
from ch03 import VLABackbone
from ch04 import (AutoregressiveActionHead, FactorizedActionHead,
                  ParallelDecodeActionHead)

backbone = VLABackbone().to(device)
backbone.language_backbone.set_attn_implementation('eager')
factorized = FactorizedActionHead().to(device)
autoregressive = AutoregressiveActionHead(backbone).to(device)
head = ParallelDecodeActionHead(backbone).to(device)
print('factorized grid:', factorized.grid)
print('AR action embeddings:', autoregressive.action_embeddings.num_embeddings)
print('parallel action positions:', head.horizon)

## 4.5 Load a real action chunk and run one forward pass

In [ ]:
from ch04.data import prepare_batch, action_targets
from ch04.backbone_adapter import encode_prefix, gather_state_hidden
from ch04.losses import masked_token_cross_entropy

batch = next(iter(validation_loader))
model_inputs = prepare_batch(batch, stats, device, backbone)
target_bins, token_pad = action_targets(
    batch, stats, tokenizer, device)

with torch.no_grad():
    prefix_hidden = encode_prefix(backbone, *model_inputs)
    state_hidden = gather_state_hidden(prefix_hidden)
    factorized_logits = factorized(state_hidden)
    parallel_logits = head(*model_inputs)
    ar_logits = autoregressive.teacher_forced_logits(
        *model_inputs, target_bins)
    loss = masked_token_cross_entropy(
        parallel_logits, target_bins, token_pad)
print('actions:', tuple(batch['action'].shape))
print('factorized:', tuple(factorized_logits.shape))
print('autoregressive:', tuple(ar_logits.shape))
print('parallel:', tuple(parallel_logits.shape))
print(f'initial CE={loss.item():.3f}; log(256)={math.log(256):.3f}')

In [ ]:
# AR inference uses a KV cache and returns the complete [B,H,D] grid.
with torch.no_grad():
    ar_grid = autoregressive.generate(*model_inputs, temperature=0.0)
print('AR first timestep bins:', ar_grid[0, 0].tolist())

## 4.5.3 Train the policy

Ten steps are a pipeline check. Set `TRAIN_STEPS = 20_000` for the manuscript run. The shipped path trains the parallel head and the trainable Chapter 3 components with separate learning rates; SigLIP remains frozen.

In [ ]:
from ch04.train import train_action_head

TRAIN_STEPS = 10  # change to 20_000 for the full experiment
history = train_action_head(
    head, backbone, train_loader, stats, tokenizer, device,
    total_steps=TRAIN_STEPS,
    warmup_steps=min(5, TRAIN_STEPS - 1),
    log_every=1,
    validation_loader=validation_loader,
    checkpoint_dir='/content/ch04-checkpoints/parallel')

## 4.6 Inspect the learned distribution

In [ ]:
from ch04.decoding import evaluation_mode, sample_logits
from ch04.diagnostics import plot_action_diagnostics, temporal_jitter

with torch.no_grad(), evaluation_mode(head):
    parallel_logits = head(*model_inputs)
expert_pairs = target_bins[:, 0, [4, 5]].cpu().numpy()
marginal, joint = plot_action_diagnostics(
    parallel_logits, expert_pairs, example=0, timestep=0,
    dims=(4, 5), support_radius=8.0)
marginal.axes[0].set_title('Policy marginal: timestep 0, control 4')
joint.axes[0].set_title('Parallel draws against held-out support')
plt.show()
sampled = sample_logits(parallel_logits, greedy=False)[0].cpu().numpy()
print('parallel temporal jitter:', temporal_jitter(sampled))

## 4.7 Decode tokens back to controls

The tokenizer returns normalized actions. The decoder then applies Chapter 2's inverse statistics so predictions and expert actions are compared in the same raw dataset units. This is open-loop validation, not a claim of safe robot deployment.

In [ ]:
import itertools
from ch04.decoding import (decode_parallel_chunk, evaluate_open_loop,
                             mean_absolute_error_by_timestep)
from ch04.diagnostics import plot_chunk_comparison

prediction = decode_parallel_chunk(
    head, model_inputs, tokenizer, stats, strategy='argmax')
expert = batch['action'].float()
pad = batch.get('action_is_pad')
if pad is not None:
    pad = pad.bool()
mae = mean_absolute_error_by_timestep(prediction, expert, pad)
print('MAE by chunk timestep:', mae.numpy().round(4))
FULL_OPEN_LOOP = False
evaluation_batches = (validation_loader if FULL_OPEN_LOOP else
                      itertools.islice(validation_loader, 4))
metrics = evaluate_open_loop(
    head, evaluation_batches, tokenizer, stats, backbone, device)
print('dataset MAE [H,D]:', metrics['mae'].shape)
print('MAE / train std:',
      metrics['mae_in_standard_deviations'].nanmean().item())
plot_chunk_comparison(prediction[0].numpy(), expert[0].numpy())
plt.show()

## Next experiments

- Train all three heads from separate fresh Chapter 3 backbones for equal step counts before interpreting their diagnostic differences.
- Train the main parallel head for 20k steps and compare checkpoint entropy.
- Compare chunk-by-chunk execution with `TemporalEnsembler` on validation episodes.
- Keep physical deployment separate: dataset units and the simulator/robot control mode must be converted and safety-checked explicitly.